In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://iabd-spark-master:7077")
    .appName("spark4-demo")

    # --- SIN Hive, catálogo de Spark ---
    .config("spark.sql.warehouse.dir", "s3a://warehouse/spark-tables")

    # --- Classpath ---
    .config("spark.driver.extraClassPath", "/opt/spark/extra-jars/*")
    .config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*")

    # --- MinIO / S3A ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://iabd-minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # --- Kafka ---
    # .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0")

    # --- Timeouts ---
    .config("spark.network.timeout", "300s")
    .config("spark.executor.heartbeatInterval", "60s")
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "2g")

    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 17:48:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.createDataFrame([(1, "test")], ["id", "valor"])
df.write.mode("overwrite").parquet("s3a://warehouse/test_directo")
print("Escritura OK")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
[Stage 0:=============================>                             (2 + 2) / 4]

Escritura OK


In [3]:
# Crear tabla persistente en MinIO (sin Hive)
spark.sql("""
    CREATE TABLE IF NOT EXISTS ventas (
        id INT, producto STRING, cantidad INT, precio DOUBLE, fecha DATE
    ) USING parquet
    LOCATION 's3a://warehouse/spark-tables/ventas'
""")

DataFrame[]

In [4]:
spark.sql("""
    INSERT INTO ventas VALUES
    (1, 'Laptop',   5,  999.99, DATE '2025-01-15'),
    (2, 'Mouse',    50, 19.99,  DATE '2025-01-16'),
    (3, 'Teclado',  30, 49.99,  DATE '2025-01-17'),
    (4, 'Monitor',  10, 349.99, DATE '2025-02-01'),
    (5, 'Webcam',   20, 79.99,  DATE '2025-02-05')
""")

DataFrame[]

In [5]:
spark.sql("SELECT * FROM ventas ORDER BY fecha").show()

+---+--------+--------+------+----------+
| id|producto|cantidad|precio|     fecha|
+---+--------+--------+------+----------+
|  1|  Laptop|       5|999.99|2025-01-15|
|  1|  Laptop|       5|999.99|2025-01-15|
|  2|   Mouse|      50| 19.99|2025-01-16|
|  2|   Mouse|      50| 19.99|2025-01-16|
|  3| Teclado|      30| 49.99|2025-01-17|
|  3| Teclado|      30| 49.99|2025-01-17|
|  4| Monitor|      10|349.99|2025-02-01|
|  4| Monitor|      10|349.99|2025-02-01|
|  5|  Webcam|      20| 79.99|2025-02-05|
|  5|  Webcam|      20| 79.99|2025-02-05|
+---+--------+--------+------+----------+



### Tipo VARIANT (datos semi-estructurados)

In [6]:
# VARIANT permite almacenar JSON semi-estructurado de forma nativa
spark.sql("""
    SELECT
        parse_json('{"nombre": "Ana", "edad": 30, "tags": ["vip", "premium"]}') AS datos,
        parse_json('{"nombre": "Luis", "edad": 25}') AS datos2
""").show(truncate=False)

# Extraer campos de VARIANT
spark.sql("""
    WITH raw AS (
        SELECT parse_json('{"producto": "Laptop", "specs": {"ram": 16, "cpu": "i7"}}') AS v
    )
    SELECT
        v:producto AS producto,
        v:specs.ram AS ram,
        v:specs.cpu AS cpu
    FROM raw
""").show()

+---------------------------------------------------+---------------------------+
|datos                                              |datos2                     |
+---------------------------------------------------+---------------------------+
|{"edad":30,"nombre":"Ana","tags":["vip","premium"]}|{"edad":25,"nombre":"Luis"}|
+---------------------------------------------------+---------------------------+

+--------+---+----+
|producto|ram| cpu|
+--------+---+----+
|"Laptop"| 16|"i7"|
+--------+---+----+



### SQL User-Defined Functions (SQL UDFs)

In [7]:
# Spark 4 permite crear UDFs directamente en SQL (sin Python/Scala)
spark.sql("""
    CREATE OR REPLACE TEMPORARY FUNCTION calcular_iva(precio DOUBLE)
    RETURNS DOUBLE
    RETURN precio * 1.21
""")

spark.sql("""
    SELECT producto, precio, calcular_iva(precio) AS precio_con_iva
    FROM ventas
""").show()

+--------+------+--------------+
|producto|precio|precio_con_iva|
+--------+------+--------------+
| Monitor|349.99|      423.4879|
|  Webcam| 79.99|       96.7879|
| Monitor|349.99|      423.4879|
|  Webcam| 79.99|       96.7879|
| Teclado| 49.99|       60.4879|
| Teclado| 49.99|       60.4879|
|  Laptop|999.99|     1209.9879|
|  Laptop|999.99|     1209.9879|
|   Mouse| 19.99|       24.1879|
|   Mouse| 19.99|       24.1879|
+--------+------+--------------+



### Pipe Syntax (operador |>)

In [8]:
# El pipe syntax permite encadenar transformaciones de forma mas legible
spark.sql("""
    SELECT * FROM ventas
    |> WHERE cantidad > 10
    |> SELECT producto, cantidad, precio * cantidad AS total
    |> ORDER BY total DESC
""").show()

+--------+--------+-----------------+
|producto|cantidad|            total|
+--------+--------+-----------------+
|  Webcam|      20|           1599.8|
|  Webcam|      20|           1599.8|
| Teclado|      30|           1499.7|
| Teclado|      30|           1499.7|
|   Mouse|      50|999.4999999999999|
|   Mouse|      50|999.4999999999999|
+--------+--------+-----------------+



### ANSI Mode por defecto

En Spark 4, `spark.sql.ansi.enabled` es `true` por defecto.
Esto significa que operaciones invalidas lanzan errores en vez de devolver NULL.

In [9]:
# Esto lanza un error en Spark 4 (en Spark 3 devolveria NULL)
try:
    spark.sql("SELECT CAST('abc' AS INT)").show()
except Exception as e:
    print(f"Error esperado (ANSI mode): {e}")

{"ts": "2026-04-12 17:49:00.220", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o113.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== SQL (line 1, position 8) ==\nSELECT CAST('abc' AS INT)\n       ^^^^^^^^^^^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionE

Error esperado (ANSI mode): [CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 1, position 8) ==
SELECT CAST('abc' AS INT)
       ^^^^^^^^^^^^^^^^^^



In [10]:
spark.stop()